In [1]:
from transformers import AutoTokenizer, AutoModel
import torch


#Mean Pooling - Take attention mask into account for correct averaging
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0] #First element of model_output contains all token embeddings
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

/Users/taraskvitko/PycharmProjects/jupiter/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
tokenizer = AutoTokenizer.from_pretrained('sentence-transformers/paraphrase-xlm-r-multilingual-v1')
model = AutoModel.from_pretrained('sentence-transformers/paraphrase-xlm-r-multilingual-v1')

Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 11908.16it/s]
XLMRobertaModel LOAD REPORT from: sentence-transformers/paraphrase-xlm-r-multilingual-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
import pandas as pd

# Open a TSV file using read_csv
df = pd.read_csv('qa-intents-dataset-university-domain/dataset_train.tsv', sep='\t')
df.columns = ['text', 'class']
df

,text,class
0,оформить справку,statement_general
1,взять справку,statement_general
2,справку как получить,statement_general
3,справку ммф где получаться,statement_general
4,справка как получить,statement_general
...,...,...
13224,тупой,smalltalk_abuse
13225,робот бестолковый,smalltalk_abuse
13226,несообразительный,smalltalk_abuse
13227,ты бестолковый,smalltalk_abuse


In [6]:
sentences = df['text'].tolist()

In [7]:
encoded_input = tokenizer(sentences, padding=True, truncation=True, return_tensors='pt')

with torch.no_grad():
    model_output = model(**encoded_input)

sentence_embeddings = mean_pooling(model_output, encoded_input['attention_mask'])

In [8]:
import nmslib

index = nmslib.init(method='hnsw', space='cosinesimil')
index.addDataPointBatch(sentence_embeddings, ids=list(range(len(sentence_embeddings))))
index.createIndex({'post': 2}, print_progress=True)


0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
*

In [38]:
test_sentence = 'что ты ел на обед'
encoded_input = tokenizer([test_sentence], padding=True, truncation=True, return_tensors='pt')
with torch.no_grad():
    model_output = model(**encoded_input)
test_sentence_embeddings = mean_pooling(model_output, encoded_input['attention_mask'])

ids, distances = index.knnQuery(test_sentence_embeddings, k=10)
for i, d in zip(ids, distances):
    print(sentences[i], '\t', d)

что угодно обедать. 	 0.21250856
неважно обедать? 	 0.22380364
где можно поесть 	 0.33236253
где поесть точку питания в главном корпусе 	 0.3353392
находится поесть в корпусе старом 	 0.34330225
находится поесть в корпусе старом 	 0.34330225
поесть в корпусе старом распологается где 	 0.35030186
поесть в корпусе старом где есть 	 0.35072482
что угодно кушать. 	 0.35409462
где поесть кафешку в главном корпусе 	 0.35840487


In [41]:
id_, _ = index.knnQuery(test_sentence_embeddings, k=1)
answer = df[df['text'] == sentences[id_.item()]]
answer['class']

5569    loc_eat
Name: class, dtype: object